# Process Mining & Conformance Analysis

**Data:** 10,000 government service requests with 52K+ lifecycle events  
**Activities:** Submitted → Triaged → Assigned → In Progress → Resolved → Closed (+ Escalated, Reopened)  
**Snowflake Features:** SQL analytics, `CORTEX.COMPLETE` for pattern discovery, Anomaly Detection on durations

In [ ]:
import snowflake.connector
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

conn = snowflake.connector.connect(connection_name=os.getenv('SNOWFLAKE_CONNECTION_NAME') or 'AU_DEMO50')
cur = conn.cursor()
cur.execute('USE DATABASE CDSB_DEMO')
cur.execute('USE SCHEMA RAW')
cur.execute('USE WAREHOUSE COMPUTE_WH')
print('Connected')

## 1. Explore the Event Log
10,000 service requests with lifecycle events — modelled on QLD government service delivery

In [ ]:
df_summary = pd.read_sql("""
    SELECT ACTIVITY, COUNT(*) as EVENT_COUNT,
           COUNT(DISTINCT CASE_ID) as CASES
    FROM SERVICE_REQUEST_EVENTS
    GROUP BY ACTIVITY
    ORDER BY EVENT_COUNT DESC
""", conn)
print(df_summary.to_string(index=False))

fig = px.bar(df_summary, x='ACTIVITY', y='CASES', color='ACTIVITY',
             title='Cases Reaching Each Activity Stage')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

## 2. Discover Process Paths
Use SQL to mine the actual process paths — find what really happens vs what should happen

In [ ]:
df_paths = pd.read_sql("""
    WITH case_paths AS (
        SELECT CASE_ID,
               LISTAGG(ACTIVITY, ' → ') WITHIN GROUP (ORDER BY EVENT_TIME) as PROCESS_PATH,
               COUNT(*) as STEPS,
               MIN(EVENT_TIME) as START_TIME,
               MAX(EVENT_TIME) as END_TIME,
               DATEDIFF('hour', MIN(EVENT_TIME), MAX(EVENT_TIME)) as DURATION_HOURS,
               MAX(CHANNEL) as CHANNEL,
               MAX(REQUEST_TYPE) as REQUEST_TYPE
        FROM SERVICE_REQUEST_EVENTS
        GROUP BY CASE_ID
    )
    SELECT PROCESS_PATH, COUNT(*) as CASES,
           ROUND(AVG(DURATION_HOURS), 1) as AVG_HOURS,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) as PCT
    FROM case_paths
    GROUP BY PROCESS_PATH
    ORDER BY CASES DESC
    LIMIT 15
""", conn)

print("Top 15 Process Paths Discovered:")
print(df_paths.to_string(index=False))

In [ ]:
fig = px.bar(df_paths.head(10), x='PROCESS_PATH', y='CASES',
             color='AVG_HOURS', color_continuous_scale='RdYlGn_r',
             title='Top 10 Process Paths (color = avg duration in hours)')
fig.update_layout(template='plotly_white', xaxis_tickangle=-20)
fig.show()

## 3. Conformance Analysis
Define the expected "happy path" and detect deviations

In [ ]:
df_conformance = pd.read_sql("""
    WITH case_paths AS (
        SELECT CASE_ID,
               LISTAGG(ACTIVITY, ' → ') WITHIN GROUP (ORDER BY EVENT_TIME) as PROCESS_PATH,
               COUNT(*) as STEPS,
               DATEDIFF('hour', MIN(EVENT_TIME), MAX(EVENT_TIME)) as DURATION_HOURS,
               MAX(CHANNEL) as CHANNEL,
               MAX(REQUEST_TYPE) as REQUEST_TYPE,
               MAX(REGION) as REGION
        FROM SERVICE_REQUEST_EVENTS
        GROUP BY CASE_ID
    )
    SELECT *,
        CASE 
            WHEN PROCESS_PATH = 'Submitted → Triaged → Assigned → In Progress → Resolved → Closed'
                THEN 'Conformant'
            WHEN PROCESS_PATH LIKE '%Escalated%' THEN 'Escalated'
            WHEN PROCESS_PATH LIKE '%Reopened%' THEN 'Reopened'
            WHEN PROCESS_PATH NOT LIKE '%Triaged%' THEN 'Skipped Triage'
            WHEN PROCESS_PATH NOT LIKE '%Closed%' THEN 'Not Closed'
            ELSE 'Other Deviation'
        END as CONFORMANCE_STATUS
    FROM case_paths
""", conn)

status_counts = df_conformance['CONFORMANCE_STATUS'].value_counts()
print("Conformance Breakdown:")
print(status_counts.to_string())

fig = px.pie(values=status_counts.values, names=status_counts.index,
             title='Process Conformance Analysis', hole=0.4)
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
fig = px.box(df_conformance, x='CONFORMANCE_STATUS', y='DURATION_HOURS',
             color='CONFORMANCE_STATUS',
             title='Resolution Time by Conformance Status (hours)')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

avg_by_status = df_conformance.groupby('CONFORMANCE_STATUS')['DURATION_HOURS'].agg(['mean', 'median', 'count'])
print("\nAverage Duration by Conformance Status:")
print(avg_by_status.round(1).to_string())

## 4. LLM-Powered Process Discovery
Ask Cortex AI to analyse the patterns and explain what it finds

In [ ]:
path_summary = df_paths.to_string(index=False)
conformance_summary = avg_by_status.round(1).to_string()

channel_analysis = df_conformance.groupby(['CHANNEL', 'CONFORMANCE_STATUS']).size().unstack(fill_value=0)
channel_text = channel_analysis.to_string()

prompt = f"""You are a process improvement consultant for a Queensland Government department 
that handles citizen service requests (licences, complaints, permits, registrations, payments).

Here are the process mining results from 10,000 service requests:

TOP PROCESS PATHS:
{path_summary}

CONFORMANCE ANALYSIS (avg duration in hours):
{conformance_summary}

CHANNEL vs CONFORMANCE:
{channel_text}

Please provide:
1. KEY FINDINGS: What are the main process patterns? What percentage follows the happy path?
2. BOTTLENECKS: Where are the biggest delays? Which deviations cause the most impact?
3. CHANNEL INSIGHTS: Do different channels (web, phone, in-person, email) have different outcomes?
4. RECOMMENDATIONS: Top 3 process improvements the department should prioritise
5. RISK FLAGS: Any patterns suggesting potential fraud, non-compliance, or customer harm?"""

llm_result = pd.read_sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-sonnet-4-6', $${prompt}$$) as ANALYSIS
""", conn)
print(llm_result.iloc[0]['ANALYSIS'])

## 5. Bottleneck Analysis — Activity Transition Times

In [ ]:
df_transitions = pd.read_sql("""
    WITH ordered_events AS (
        SELECT CASE_ID, ACTIVITY, EVENT_TIME,
               LEAD(ACTIVITY) OVER (PARTITION BY CASE_ID ORDER BY EVENT_TIME) as NEXT_ACTIVITY,
               LEAD(EVENT_TIME) OVER (PARTITION BY CASE_ID ORDER BY EVENT_TIME) as NEXT_TIME
        FROM SERVICE_REQUEST_EVENTS
    )
    SELECT ACTIVITY || ' → ' || NEXT_ACTIVITY as TRANSITION,
           COUNT(*) as FREQUENCY,
           ROUND(AVG(DATEDIFF('hour', EVENT_TIME, NEXT_TIME)), 1) as AVG_HOURS,
           ROUND(MEDIAN(DATEDIFF('hour', EVENT_TIME, NEXT_TIME)), 1) as MEDIAN_HOURS,
           ROUND(MAX(DATEDIFF('hour', EVENT_TIME, NEXT_TIME)), 1) as MAX_HOURS
    FROM ordered_events
    WHERE NEXT_ACTIVITY IS NOT NULL
    GROUP BY TRANSITION
    ORDER BY AVG_HOURS DESC
""", conn)

print(df_transitions.to_string(index=False))

fig = px.bar(df_transitions, x='TRANSITION', y='AVG_HOURS',
             color='FREQUENCY', color_continuous_scale='Blues',
             title='Average Time Between Activities (hours)')
fig.update_layout(template='plotly_white', xaxis_tickangle=-30)
fig.show()

## 6. Prepare Data for Neo4j Graph Analytics
Generate node and relationship tables for graph-based process analysis

In [ ]:
cur.execute("""
    CREATE OR REPLACE TABLE PROCESS_NODES AS
    SELECT ROW_NUMBER() OVER (ORDER BY NODE_TYPE, NODE_ID) as nodeId,
           NODE_ID, NODE_TYPE
    FROM (
        SELECT DISTINCT CAST(CASE_ID AS VARCHAR) as NODE_ID, 'Case' as NODE_TYPE
        FROM SERVICE_REQUEST_EVENTS
        UNION ALL
        SELECT DISTINCT ACTOR as NODE_ID, 'Handler' as NODE_TYPE
        FROM SERVICE_REQUEST_EVENTS
        UNION ALL
        SELECT DISTINCT REGION as NODE_ID, 'Region' as NODE_TYPE
        FROM SERVICE_REQUEST_EVENTS
    )
""")

cur.execute("""
    CREATE OR REPLACE TABLE PROCESS_RELATIONSHIPS AS
    SELECT n1.nodeId as sourceNodeId, n2.nodeId as targetNodeId, 
           'HANDLED_BY' as relationshipType, COUNT(*) as weight
    FROM SERVICE_REQUEST_EVENTS e
    JOIN PROCESS_NODES n1 ON CAST(e.CASE_ID AS VARCHAR) = n1.NODE_ID AND n1.NODE_TYPE = 'Case'
    JOIN PROCESS_NODES n2 ON e.ACTOR = n2.NODE_ID AND n2.NODE_TYPE = 'Handler'
    GROUP BY n1.nodeId, n2.nodeId

    UNION ALL

    SELECT n1.nodeId, n2.nodeId, 'IN_REGION', COUNT(*)
    FROM SERVICE_REQUEST_EVENTS e
    JOIN PROCESS_NODES n1 ON CAST(e.CASE_ID AS VARCHAR) = n1.NODE_ID AND n1.NODE_TYPE = 'Case'
    JOIN PROCESS_NODES n2 ON e.REGION = n2.NODE_ID AND n2.NODE_TYPE = 'Region'
    GROUP BY n1.nodeId, n2.nodeId
""")

nodes = pd.read_sql("SELECT NODE_TYPE, COUNT(*) as n FROM PROCESS_NODES GROUP BY NODE_TYPE", conn)
rels = pd.read_sql("SELECT relationshipType, COUNT(*) as n FROM PROCESS_RELATIONSHIPS GROUP BY relationshipType", conn)
print("Nodes:", nodes.to_string(index=False))
print("\nRelationships:", rels.to_string(index=False))
print("\n→ Ready for Neo4j Graph Analytics (next notebook)")

## Key Takeaways

| Capability | Approach | No Specialist Tools Needed |
|-----------|----------|---------------------------|
| Process Discovery | SQL `LISTAGG` + `LEAD/LAG` | Just SQL windowing functions |
| Conformance Checking | Path comparison with `CASE` | Pattern matching in SQL |
| Bottleneck Analysis | `DATEDIFF` on transitions | Standard SQL |
| Pattern Explanation | `CORTEX.COMPLETE` | LLM interprets data inline |
| Graph Preparation | Node/relationship tables | Ready for Neo4j or native graph |

**Process mining doesn't require specialised tools — SQL + Cortex AI gets you 80% of the way**